In [ ]:
import pandas as pd
from scipy.io import arff

base_path = "/content/drive/MyDrive/nsl_kdd/"

train_path = base_path + "KDDTrain+.txt"
test_path = base_path + "KDDTest+.txt"

nsl_train = pd.read_csv(train_path, header=None)
nsl_test = pd.read_csv(test_path, header=None)

print("Train shape:", nsl_train.shape)
print("Test shape:", nsl_test.shape)

column_names = [
'duration','protocol_type','service','flag','src_bytes','dst_bytes','land',
'wrong_fragment','urgent','hot','num_failed_logins','logged_in',
'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
'num_shells','num_access_files','num_outbound_cmds','is_host_login',
'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
'dst_host_same_srv_rate','dst_host_diff_srv_rate',
'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
'dst_host_serror_rate','dst_host_srv_serror_rate',
'dst_host_rerror_rate','dst_host_srv_rerror_rate',
'class','difficulty'
]

nsl_train.columns = column_names
nsl_test.columns = column_names

nsl_train.head()

print("Unique classes:", nsl_train['class'].nunique())
print(nsl_train['class'].value_counts().head(10))


# Define attack category mapping

dos_attacks = ['neptune','smurf','back','teardrop','pod','land']
probe_attacks = ['satan','ipsweep','nmap','portsweep']
r2l_attacks = ['warezclient','guess_passwd','warezmaster','ftp_write','imap','multihop','phf','spy']
u2r_attacks = ['buffer_overflow','loadmodule','perl','rootkit']

def map_attack(label):
    if label == 'normal':
        return 'NORMAL'
    elif label in dos_attacks:
        return 'DOS'
    elif label in probe_attacks:
        return 'PROBE'
    else:
        return 'R2L_U2R'

nsl_train['attack_category'] = nsl_train['class'].apply(map_attack)
nsl_test['attack_category'] = nsl_test['class'].apply(map_attack)

print(nsl_train['attack_category'].value_counts())


nsl_train['attack_category'].value_counts()

# Normal-only dataset
nsl_train_normal = nsl_train[nsl_train['attack_category'] == 'NORMAL']

# Attack-only dataset (for later evaluation)
nsl_train_attack = nsl_train[nsl_train['attack_category'] != 'NORMAL']

print("Normal samples:", nsl_train_normal.shape)
print("Attack samples:", nsl_train_attack.shape)

nsl_train = nsl_train.drop(columns=['class', 'difficulty'])
nsl_test = nsl_test.drop(columns=['class', 'difficulty'])

nsl_train.to_csv("/content/drive/MyDrive/nsl_kdd/nsl_kdd_train_processed.csv", index=False)
nsl_test.to_csv("/content/drive/MyDrive/nsl_kdd/nsl_kdd_test_processed.csv", index=False)

print("Phase 1 complete: Clean dataset saved.")


Train shape: (125973, 43)
Test shape: (22544, 43)
Unique classes: 23
class
normal         67343
neptune        41214
satan           3633
ipsweep         3599
portsweep       2931
smurf           2646
nmap            1493
back             956
teardrop         892
warezclient      890
Name: count, dtype: int64
attack_category
NORMAL     67343
DOS        45927
PROBE      11656
R2L_U2R     1047
Name: count, dtype: int64
Normal samples: (67343, 44)
Attack samples: (58630, 44)
Phase 1 complete: Clean dataset saved.
